***
# AirBnB Listings in Zurich: Data analysis
***
Before delving into the analysis of Airbnb listings in the city of Zurich, let's iinstall all necessary libraries:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import geopandas as gpd
import folium
import re
import statsmodels.api as sm
from shapely.geometry import Point
from scipy import stats
from esda.moran import Moran

In [ ]:
import os
os.makedirs("images", exist_ok=True)

In [ ]:
%load_ext autoreload
%autoreload 2

***
## 1. About the Project
### 1.1. AirBnB listings as a topic
The short-term rental market, AirBnB in particular, has grown rapidly in urban areas, influencing rents, local economies, and urban planning. In cities like Zürich, understanding the factors of Airbnb prices can provide insights into market dynamics, potential regulatory interventions, and correlations with traditional rental prices. 
### 1.2. The Datasets
Inside AirBnB, the provider of our main dataset, is a non-commercial third-party provider of AirBnB data that aims to increase transparency of Airbnb activity. They scrape the official Airbnb website regularly and structure it csv files which can be obtained via their website https://insideairbnb.com/.
In addition, we are using two other datasets, one for normalizing the airbnb listings over neighborhoods in Zurich and the other to compare these normalized listing quantities with rental prices in each neighborhood.
### 1.3. Our Goals
In our project we aim to analyse Airbnb listings in Zürich, identify key features influencing pricing, and compare them with municipal rental statistics.
***
## 2. Loading, Cleaning and Validating the datasets
### 2.1. Loading and displaying dataframe overviews


In [ ]:
# define an overview function
def overview(df, name="dataset"):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True)
    }).sort_values(["dtype", "missing_pct"], ascending=[True, False])

    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    return summary

#loading and displaying
## Airbnb
listings_path = 'data/listings.csv'
airbnb_df= pd.read_csv(listings_path, encoding="utf-8")
print(overview(airbnb_df, "Airbnb"))

## housing stock
housing_path = 'data/bau522od5221_wohnungsbestand_zurich.csv'
housing_df = pd.read_csv(housing_path, encoding="utf-8")
display(overview(housing_df, "Housing"))

## Rental prices
rental_path = 'data/rental_prices.csv'
rental_df = pd.read_csv(rental_path, encoding="utf-8")
display(overview(rental_df,"Rental"))

### 2.2. Cleaning
Based on the overview, we have decided to clean the datasets the following way:
- Generally:
    - Standardize column names (lowercase, underscores etc.).
    - Turn every column with 2 unique values to boolean.
    - All object data types are converted to more primitive dtypes (string, float, int, category, sorted category).
- Airbnb:
    - Remove all listings without a price (since this variable is imperative for later analysis)
    - Drop unnecessary columns (i.e. everything with a URL).
    - Remove columns with too much unusable noise, like *host_description* in *airbnb_df*. Instead, we added a column that states the length of the entry.
    - Reference all dates or temporal variables to the date the data was scraped.
    - Feature engineering amenities in teh Airbnb dataframe by extracting amenities from the list in the *amenities* column.
- Housing Stock:
    - Only keep entries from 2025
    - Restructure the dataframe to only have 32 rows(one for each neighborhood). Aggregate the number of objects into separate columns for different apartment sizes.
- Rental Prices:
    - Only keep entries for 2024 (remove 2022)
    - Only keep square meter entries
    - Only keep netto entries (to remove differences in accounting etc. between neighborhoods as much as possible)
    - Restructure the dataframe to only have 32 rows (one for each neighborhood). The columns now represent different rental prices.

A unique normalization function was defined for each of the 3 datasets in the file *normalize.py*. Consult this file for further detail on the normalization process.

In [ ]:
from src.normalize import normalize_airbnb, normalize_housing, normalize_rental

# normalize Airbnb df
airbnb_df_norm = normalize_airbnb(airbnb_df)

# normalize housing stock df
housing_df_norm = normalize_housing(housing_df, year=2025)

# normalize rental price df
rental_df_norm = normalize_rental(rental_df, year=2024, brutto=False, sqm=True, level = 5, cat_zimmer=False)

***
## 3. Aggregating to quartier-level
At this point, the housing stock dataframe and rental price dataframe contain hundreds of rows, where each Quartier has several entries. For our analysis on quartier-level, we prefer a different structure, where each quartier only has one row. For that reson, we create a new pivot table around the quartier-column.
### 3.1. Using Pivot Tables

In [ ]:
from src.aggregation import quartiere_standardized_housing, quartiere_standardized_rental

# housing stock
housing_df_quart = quartiere_standardized_housing(housing_df_norm)
# flatten columns of housing df to 1:
housing_df_quart.columns = [
    "_".join(col).strip() if isinstance(col, tuple) else col
    for col in housing_df_quart.columns
]
housing_df_quart = housing_df_quart.drop(columns="index_")
display(housing_df_quart.head())

# rental prices
rental_df_quart = quartiere_standardized_rental(rental_df_norm)
display(rental_df_quart.head())

Now, the two datasets are of length 35 (i.e. both only contain one row for each of the 34 quartiere).
### 3.2. Add Airbnb count to Housing Stock dataframe
In order to compare airbnb listings (normalized by housing stock) and rental prices in each quartier, we need to know how many listings are in each quartier. Luckily, INside Airbnb provides the quartier that each listing lies in, so we simply need to count them and merge that count to *housing_df_quart*:

In [ ]:
# count airbnbs per quartier
counts_airbnb_quartier = (
    airbnb_df_norm
    .groupby("neighbourhood_cleansed")
    .size()
    .rename("n_airbnbs")
)

# merge
housing_df_quart = housing_df_quart.merge(
    counts_airbnb_quartier,
    left_on="quarlang_",
    right_index=True,
    how="left"
)


***
## 4. Exploratory Data Analysis
### 4.1. Airbnb Dataset
#### 4.1.1. Price
Let's simply calculate the minimum and maximum price for a listing in the dataset:

In [ ]:
minimum = min(airbnb_df_norm["price"])
maximum = max(airbnb_df_norm["price"])
print(f"Airbnb prices:\nMin: {minimum}\nMax: {maximum}")

Now, let us take a look at the distribution of listing prices in the dataset. We generate a histogram and a boxplot. Since there are some heavy outliers with very high prices, let us also crop the two plots to inspect the distribution at 'normal' price ranges (between 0 and 1000).

In [ ]:
print(max(airbnb_df_norm["price"]))

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize = (10,7))

ax1.hist(airbnb_df_norm["price"], bins = 100)
ax1.set_xlabel("price")
ax1.set_ylabel("count")
ax1.set_title("Histogram 'price' (bin=100)")

sns.boxplot(ax=ax2, data=airbnb_df_norm, y="price")
ax2.set_title("Boxplot 'price'")

ax3.hist(airbnb_df_norm["price"], bins = 1000)
ax3.set_xlim(0,1000)
ax3.set_xlabel("price")
ax3.set_ylabel("count")
ax3.set_title("Histogram 'price' cropped (bin=1000)")

sns.boxplot(ax=ax4, data=airbnb_df_norm, y="price")
ax4.set_ylim(0,500)
ax4.set_title("Boxplot 'price' cropped")

plt.tight_layout()
plt.show()

These plots show the following:
- Positive skew: mostly prices between 0 and 500, but outliers all the way at 10000
    - i.e. many cheap and moderately priced listings with few very expensive listings.
    - This means that we should be looking at the median rather than the mean when trying to get one metric for price.
    - These outliers are most likely real listings, since a market for luxury airbnb listings does exist.
- There is a singular peak: no submarkets visible (e.g. high demand for luxury listings could have resultet in a small peak at higher prices).
- 25th and 75th quantile lie at roughly 100 and 200 CHF respectively.
#### 4.1.2. Reviews
There are 7 variables for different review scores. Since we aim to use some of them in our analysis later on, it would be helpful to see how their values are structures.

In [ ]:
cols = ["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin", "review_scores_communication", "review_scores_location", "review_scores_value"]

fig, ax = plt.subplots(figsize=(12,4))
sns.boxplot(ax=ax, data=airbnb_df_norm[cols],
            order=cols
            )
ax.set_xticks(range(7))
ax.set_xticklabels(["Overall", "Accuracy", "Cleanliness", "Checkin", "Communication", "Location", "Value"])
ax.set_ylim(1,5)
plt.tight_layout()
plt.show()

As expected, all review categories are mostly positive, with medians around 4.7. Of course, there are also outliers toward the bottom, as can be expected when asking for user feedback. We are especially interested in the last variable: Value since we will try to predict those values. We can see, that it has by far the lowest mean and one of the largest variances. We can tell by this shape, that it is heavily left-skewed. We will probably have to transform it in order to use it in a regression.
#### 4.1.3. Geographic distribution
Since each listing contains values for longitude and latitude, we can plot them on a map to get a sense of how they are distributed geographically across the city. We will add the borders of the quartiere aswell as colour the listings according to their corresponding quartier.  
For this process, we need to temporarily transform our airbnb listings dataframe into a geodataframe using geopandas:

In [ ]:
# convert airbnb_df_norm to geo data frame
airbnb_gdf = gpd.GeoDataFrame(
    airbnb_df_norm,
    geometry=gpd.points_from_xy(airbnb_df_norm["longitude"], airbnb_df_norm["latitude"])).set_crs(epsg=4326)

# import quartiere borders and forested areas
filepath_quart = "data/zurich_quartiere.gpkg"
filepath_for = "data/forest_zurich.gpkg"
layer_quart = "stzh.adm_statistische_quartiere_map"
quartiere = gpd.read_file(filepath_quart, layer=layer_quart).to_crs(epsg=4326)
forest = gpd.read_file(filepath_for).to_crs(epsg=4326)

# create map with function from src/maps.py
from src.maps import zurich_map_eda
zurich_map = zurich_map_eda(airbnb_gdf, forest, quartiere)
zurich_map

By **hovering over the empty land in the quartier**, the name of the quartier shows up. The color ramp uses the log-transform of the price, so the legend is rather vague. We mainly use the colors to convey the general distribution of listing prices. By hovering over a listing, one can see the actual price which should give an idea of how the color ramp works.
We can see that airbnb **listings are clustered along the lake shore and in the city center**. Forested areas obviously do not contain any listings, but listings reach the edge basically anywhere else.  
Concerning the **price**, we can see that prices are generally **higher in the city center and lower at the municipalitites borders**, although clusters of high prices can be found outside the center like on the lake shores or in Höngg. This price distribution will become important later on when we will try to train a model which will **predict listing prices based on a set of parameters**. Looking at this map, we guess that **location could play a major part in that model**.
#### 4.1.4. Correlation of dataset variables
Since we will be using several variables from the Airbnb dataset to predict things, it could be useful to see how these variables correlate with each other. All boolean variables have been converted back to 1s and 0s for this part. Then, we only took the numeric variables for the correlation matrix and dropped a bunch of variables we think do not fit for this visualization and would simply clutter the graph.  
Aigain, we are using a function for plotting the correlation matrix which is defined in our `src` folder.

In [ ]:
from src.plots import prepare_corr_df, plot_corr_heatmap

airbnb_corr = prepare_corr_df(airbnb_df_norm)
plot_corr_heatmap(airbnb_corr)

Take-aways from this correlation plot:
- obvious correlation between similar variables like `availability_eoy`and other `availability_xxx`variables. Or all the review scores among themselves.
- `host_is_superhost` has medium correlation with quite a lot of other variables (at least compared to most other pairings).
- Surprisingly to us, `price` does not show even moderate correlation with any variable except `estimated_revenue_l365d` (latter seems to be a function of the prior anyways). Knowing this, we expect it will be hard to find a set of variables which can be used to train a reliable model to predict listing price, however we can still consult this correlation matrix to get a starting set of variables, which we think might have the highest influence on the model.
- In general, negtive correlation between variables is much less common and less extreme than positive correlation. This means that there is no variable that causes another variable to decrease significantly if itself increased.
### 4.2. Housing Stock Dataset
#### 4.2.1. Basic statistics of housing stock dataset
First, let us look at how the different stock-sizes (e.g. `_1-Zimmer`or `_5-Zimmer`) are distributed across the quartiere with boxplots:

In [ ]:
# restructure to get columns by number of rooms
from src.plots import restructure_housing_by_rooms, box_and_stacked_housing_stock
housing_rooms = restructure_housing_by_rooms(housing_df_quart)

# normalize for each quartier
housing_share = housing_rooms.div(housing_rooms.sum(axis=1), axis=0)
# plotting using src/plots.py
box_and_stacked_housing_stock(housing_share)

What we can gather from the boxplot:  
- Objects with 3 rooms are the most common. This follows typical housing structure patterns where the most common sizes are 3 or 4 rooms and they get less common towards the extremes like 1 room or 6 rooms.
- There is noticeable difference in variance (e.g. larger variance for 1 room than for 5 rooms), but these differences are not extreme.
- There are a couple of outliers which we can inspect in the stacked bar chart to the right.
- The two outliers from the boxplot for 1 room are now visible: Hochschulen and Rathaus have significantly more stock with 1 room than the others. This might be due to more student housing in this area.  
- The two outliers for 6 rooms are Hottingen and Fluntern, both quartiere lie on the slopes of the Zürichberg. It makes sense that one would find larger estates there, since they are considered more rich parts of the city.  
#### 4.2.2. Geographic distribution of airbnb stock
In order to get an idea of where the airbnb listings are situated, not on the level of each individual listing, but on the level of the quartiere, we can create a simple choropleth map. But first, we need to join the airbnb counts from `housing_df_quart` with the gpkg file containing the quartiere and their boundaries:
- import quartiere as gdf
- merge the number of airbnbs to the gdf
- compute airbnbs per square kilometer
- plot the sorted gdf

In [ ]:
# import the quartiere gpkg
filepath_quart = "data/zurich_quartiere.gpkg"
layer_quart = "stzh.adm_statistische_quartiere_map"
quartiere_gdf = gpd.read_file(filepath_quart, layer=layer_quart).to_crs(epsg=2056)

# perform join by attribute: quarlang_
quartiere_airbnb_gdf = quartiere.merge(
    housing_df_quart[["quarsort_", "n_airbnbs"]],
    left_on="qnr",
    right_on="quarsort_",
    how="left"
).to_crs(epsg=2056)

# calculating airbnb
quartiere_airbnb_gdf["area_km2"] = quartiere_airbnb_gdf.geometry.area / 1000000
quartiere_airbnb_gdf["airbnb_km2"] = quartiere_airbnb_gdf["n_airbnbs"] / quartiere_airbnb_gdf["area_km2"]

# set index
if quartiere_airbnb_gdf.index.name != "qname":
    quartiere_airbnb_gdf = quartiere_airbnb_gdf.set_index("qname")

# sort
quartiere_airbnb_gdf = quartiere_airbnb_gdf.sort_values("airbnb_km2", ascending=False)

from src.plots import plot_per_quartier
plot_per_quartier(quartiere_airbnb_gdf, "airbnb_km2", "Airbnb listings per sqkm", "share of housing stock")

We can also plot this on a map:

There are some similarities in which quartiere are at the top of this ranking when comparing it to the previous graph. Though this correlation must be investigated further, since it could just be an artefact of these quartiere being closer to the city center than others (e.g. Rathaus, Werd, Sihlfeld, Langstrasse etc.)
### 4.3. Rental Prices Dataset
#### 4.3.1. Price for each quartier
To get a broad overview of how rental prices differ accross the quartiere in Zurich, we just plot the mean rent for each quartier.

In [ ]:
# setting index to quartier name
if rental_df_quart.index.name != "gliederunglang":
    rental_df_quart = rental_df_quart.set_index("gliederunglang")

rental_df_quart = rental_df_quart.sort_values("rent_mean_both", ascending=False)

# plot
plot_per_quartier(rental_df_quart, "rent_mean_both", "Rent in each quartier", "average rent per sqm [CHF]", grid=True)

#### 4.3.2. Mapping rental prices
To map them, we first need to join the prices to the gdf that contains the quartiere boundaries:

In [ ]:
# load gpkg as gdf of quartiere boundaries
filepath_quart = "data/zurich_quartiere.gpkg"
layer_quart = "stzh.adm_statistische_quartiere_map"
quartiere_gdf = gpd.read_file(filepath_quart, layer=layer_quart).to_crs(epsg=2056)


# join rental prices by common attribute
quartiere_rent_gdf = quartiere.merge(
    rental_df_quart.set_index("gliederungsort")[["rent_mean_both"]],
    left_on="qnr",
    right_on="gliederungsort",
    how="left"
).to_crs(epsg=2056)

# map
from src.maps import quartier_map
quartier_map(quartiere_rent_gdf, "rent_mean_both", "Average rent per sqm in Zurich [CHF]")


- The highest rents are in the center of the city which is to be expected:
    - Lindenhof:	29.54
    - Rathaus:  	27.85
    - Hochschulen:	26.21
    - Hottingen:    25.85
    - Seefeld:  	25.34
- Some of these quartiere do also have high shares of airbnb listings, like Rathaus and Lindenhof.  

This concludes the EDA of the three datasets. The following parts will focus on more in depth analysis and model training.

***
## 5. Comparative Analysis
### 5.1. Motivation
In this part of the project, we finally combine the Airbnb dataset with the data for rental prices in Zurich's quartiere. By doing this, we try to find out whether there is a correlation between airbnb listing density and rental prices among the 34 quartiere in the city.  
Such an analsyis is motivated by current discussions about airbnb bans in city centers. The argument is that short-term rentals like the offers on airbnb, artificially reduce the housing supply in an already competitive market, potentially resulting in higher rental prices overall (for more detail on the topic, please refer to [Garcia-López et al., 2020](https://doi.org/10.1016/j.jue.2020.103278)).
Henceforth, we will test for the following Hypothesis:  
*$H_1$: Higher Airbnb density (normalised by total housing stock) in the City of Zurich's quartiere is correlated with higher average rental prices.*
### 5.2. Feature engineering
We have already prepared the dataset for this task: In step *3. Aggregating to quartier-level* we aggregated the rental prices and housing stock to the quartier level and added the number of airbnbs per quartier.
Hence, for this analysis, we will be using the following dataframes:

- `housing_df_quart`
- `rental_df_quart`  

We start by merging the two dataset for easier use later:

In [ ]:
rental_airbnb_df = rental_df_quart.reset_index().merge(
    housing_df_quart[["quarlang_","total_units_", "n_airbnbs"]],
    left_on="gliederunglang",
    right_on="quarlang_",
    how="left"
)
display(rental_airbnb_df.head(1))

Now we want to engineer a new variable which normalises the number of airbnbs in each quartier. Instead of calculating airbnbs per square kilometer, we are going to calculate airbnbs per total housing stock. This takes into account larger quartiere whose area is made up in large parts of forest, which would falsify our density estimates.

In [ ]:
rental_airbnb_df["airbnb_density"] = rental_airbnb_df["n_airbnbs"] / rental_airbnb_df["total_units_"]
display(rental_airbnb_df.head(1))

### 5.3. Univariate Linear Regression
#### 5.3.1. Initial model
To assess the suitability of the linear regression model for analyzing the relationship between Airbnb density and rental prices, we examine key model assumptions:
- Linearity: We assess whether the relationship between the predictor (Airbnb density) and response variable (rental price) is approximately linear using residuals vs. fitted value plots.
- Normality of residuals: We inspect whether residuals are approximately normally distributed using a Q–Q plot.
- Homoscedasticity: We evaluate whether the variance of residuals is constant across fitted values using scale-location plots.  

These diagnostics help assess whether the linear model is an appropriate approximation of the underlying relationship, particularly for inference purposes.  

In addition to the three plots above we also plot the influence of individual points to detect severe outliers.

In [ ]:
from src.comparative import plot_fit_resid

# predictor variable and add intercept column
X = sm.add_constant(rental_airbnb_df["airbnb_density"])
# response variable
y = rental_airbnb_df["rent_mean_both"]

plot_fit_resid(X, y)

These 5 plots show, whether our data meets the assumptions. And it mostly does not:
- There is a **non-linear relationship** between X and y indicated in plot 2)
- Plot 3) shows **non-normal distribution** of the variables
- The variables are also **heteroscedastic**, which can be seen in plot 4)
- There are **4 points with influence** byond cook's distance (which has been set to 4/n with n=34), one of which, point 2, has exceptionally large influence  

We can try to refit the model after transforming the predictor variable. For this transformation we will try the logarithm which should spread the predictor variables more evenly along the x-axis, resulting in less influence, more homoscedasticity and if we are lucky, even improve the linearity of the relationship:
#### 5.3.2. Log-transforming the predictor variable

In [ ]:
# predictor variable and add intercept column
X = np.log(rental_airbnb_df["airbnb_density"])
X = sm.add_constant(X)
# response variable
y = rental_airbnb_df["rent_mean_both"]

plot_fit_resid(X, y)

That worked decently well. First, what is still bad:
- We still **lack strong linearity** as shown in plot 2) with the concave curve
- Our Q-Q plot still shows a **positive skew** 

What improved?
- The large **influence of point 2 disappeared**, as it now lies directly on the regression line (plot 1) on the very right). And the remeining influencial points are far less grave: the range of the x-axis might be misleading, as it now only stretches to 0.2, whereas before it went all the way to 0.6
- Much **more homoscedasticity** as indicated by the rather horizontal line in plot 4)  

These diagnostic plots have indicated that a simple linear model does not perfectly capture the relationship between Airbnb density and rental prices. A log-transformation of the predictor substantially improved leverage and heteroscedasticity issues, although some residual nonlinearity remains visible in the residual diagnostics.  
Given that the objective of this analysis is primarily to examine correlation the transformed linear regression is still considered informative for deriving broad insights about the relationship between airbnb density and rental prices. The results should therefore be interpreted as indicative of association patterns rather than as evidence of a perfectly specified linear relationship.  

#### 5.3.3. Analysis with bootstrapped Confidence Interval
**For reference, take the output summary of our last block of code.**  
Due to our small sample size of n=34, standard asymptotic inference may overstate the precision of our estimates. To better quantify this uncertainty, we use bootstrapping with 10,000 resamples to compute confidence intervals for the regression coefficients and R². Unlike standard OLS standard errors, bootstrapped CIs make no distributional assumptions and more honestly reflect the variability inherent in a small sample.

In [ ]:
from src.comparative import bootstrap_ols
boot_coefs, boot_r2, ci_coefs, ci_r2 = bootstrap_ols(X, y)

print("\nBootstrapped 95% CIs")
print(f"  Intercept: [{ci_coefs[0][0]:.3f}, {ci_coefs[1][0]:.3f}]")
print(f"  Slope:     [{ci_coefs[0][1]:.3f}, {ci_coefs[1][1]:.3f}]")
print(f"  R²:        [{ci_r2[0]:.3f}, {ci_r2[1]:.3f}]")

Since the **predictor variable is log-transformed**, the coefficient can be interpreted as the expected **change in mean rent associated with a one-unit increase in log(Airbnb density)**, corresponding approximately to a percentage change in Airbnb density.
Our univariate OLS regression indicates a significant positive association between the log(Airbnb-density) and mean rent (β = 2.16, p < 0.001). The bootstrapped CI confirms the significance (0 lies outside the CI of [1.417, 3.033]). The model explains 57.7% of the variation in rent (R² = 0.577) and is overall statistically significant (F-test p < 0.001), though this estimate should be interpreted cautiously given the small sample size. Once again, we should take a look at our bootstrapped CIs: The R² CI of [0.39, 0.75] reveals considerable uncertainty around the point estimate of 0.577 — the true explanatory power of the model could plausibly range from modest to strong. These results still suggest that higher Airbnb density is associated with higher rental prices in the sample. We therefore reject $H_0$ (no correlation between `log(airbnb_density)` and `rent_mean_both`) in favour of $H_1$.
#### 5.3.4. Moran's I of naive correlation
Since we are working with spatial data, we must keep in mind the possibility of spatial autocorrelation (i.e. entities that are close together have higher influence on values of their neighbours than entities that are far apart).  

To assess this, we compute Moran's I on the residuals of the regression between rental prices and log(Airbnb density).  
The null hypothesis of Moran’s I is that residuals are spatially randomly distributed. A significant result would indicate spatial co-structure of residualized rent and Airbnb density, suggesting that the model does not fully capture spatial dependence in rental prices.  

While we used bootstrapping in previous steps to account for our small sample size, significance for Moran's I was assessed via 999 random permutations rather than asymptotic approximation. This is more appropriate than standard p-values at small sample sizes (n=34), as it makes no distributional assumptions about the test statistic.

In [ ]:
from src.comparative import get_morans_weights, moran

# predictor variable and add intercept column
X = np.log(rental_airbnb_df["airbnb_density"])
# response variable
y = rental_airbnb_df["rent_mean_both"]

quartiere_gdf = quartiere_gdf.sort_index().reset_index(drop=True)
w= get_morans_weights(quartiere_gdf)
np.random.seed(42)

moran_rent = Moran(y, w)
moran_airbnb = Moran(X, w)

results = {
        "moran_rent_I": moran_rent.I,
        "moran_rent_p": moran_rent.p_sim,
        "moran_airbnb_I": moran_airbnb.I,
        "moran_airbnb_p": moran_airbnb.p_sim,
    }

results_df = pd.DataFrame([results])

results_df


The Moran’s I results indicate that rental prices exhibit significant positive spatial autocorrelation across Zurich’s quartiere (I = 0.161, p = 0.009), meaning that neighboring quartiere tend to have similar rental price levels. In contrast, Airbnb density does not show significant global spatial autocorrelation (I = 0.010, p = 0.259), suggesting that Airbnb density is not strongly organized along a city-wide spatial gradient.

This distinction is important for interpreting the regression results. The significant spatial autocorrelation in rents implies that the assumption of independent observations underlying standard OLS regression may be violated. At the same time, the lack of strong global autocorrelation in Airbnb density suggests that the observed correlation between Airbnb density and rent may partly reflect the broader spatial organization of rental prices rather than a fully shared spatial structure between both variables.
### 5.4. Confounding variable?
We need to be careful with jumping to conclusions based on the univariate linear regression above, since this model only suggests correlation between the two variables. This does not mean that higher Airbnb density actually causes higher rent. It could be that both variables are **linked to a common third variable** that affects them in the same direction. While we cannot directly identify such a variable, we can examine whether distance to the city center is associated with both the predictor and response variables in a way **consistent with confounding**. This step is also motivated by the significant spatial co-structure of residualized rent and Airbnb density which Moran's I detected in our data set in the previous step.

We will use the following approach to explore this possibility (steps 1 and 2 are already handled in the previous chapter):
1) Start with a naive (univariate) linear regression of rent on Airbnb density → yields a significant positive correlation (r = 0.77, R² = 0.57)
2) Hypothesize that both variables are driven by a common confounder: distance to city center  

New steps:  

3) **calculate distance to city center** for all quartiere (once for each of the 4 different options for city center)
4) **Partial out the confounder** by regressing each variable (rent, Airbnb density) separately on distance to center and extracting residuals
5) **Correlate the residuals** → gives the partial correlation between rent and Airbnb density with distance removed
6) **Repeat for 4 city center definitions** as a sensitivity analysis, since no single definition is ground truth
7) Compare **naive r vs. partial r** across all 4 definitions to assess whether the original finding was spurious or robust  

#### 5.4.1. Calculating distance to city center
For this next step, we first need to calculate the distance of each quartier to the city center, but where is the city center?
- geographic center point of the City of Zurich
- cultural center (e.g. Grossmünster or Bellevue)
- economic center (e.g. Paradeplatz)
- etc.  

In order to cover various definitions for city center, we will use **Zurich main station (Bahnhofplatz), Paradeplatz, Bellevue or Grossmünster** (these 4 options include financial hubs, transit hubs, cultural hubs etc.) as the central points.

In [ ]:
# load gpkg as gdf of quartiere boundaries
filepath_quart = "data/zurich_quartiere.gpkg"
layer_quart = "stzh.adm_statistische_quartiere_map"
quartiere_gdf = gpd.read_file(filepath_quart, layer=layer_quart).to_crs(epsg=2056)


# join rental prices by common attribute
rental_airbnb_gdf = quartiere.merge(
    rental_airbnb_df.set_index("gliederungsort"),
    left_on="qnr",
    right_on="gliederungsort",
    how="left"
).to_crs(epsg=2056)

# convert coordinates of these points to a gdf
from src.comparative import get_center_gdf

center_data = {
        "name": ["hb", "paradeplatz", "bellevue", "grossmunster"],
        "geometry": [
            Point(2683129, 1247959),   # Zürich HB 
            Point(2683107, 1247122),   # Paradeplatz 
            Point(2683572, 1246820),    # Bellevue 
            Point(2683464, 1247186)     # Grossmünster
            ]
    }

center_gdf = get_center_gdf(rental_airbnb_gdf, center_data)

#### 5.4.2. Partial confounder
In order to remove the effect that proximity to city center has on both the predictor variable `airbnb_density` and the response variable `rent_mean_both`, we can extract the residuals of univariate linear regressions `rent_mean_both ~ distance` and `airbnb_density ~ distance` (of course done for all 4 center options). This will leave us with `resid_rent` and `resid_airbnb` which are the parts of the predictor/response variable which cannot be explained by `dist_xx`. We then calculate the correlation between these residuals with a simple pearson r.   

Since we are now working with pearson r, we first recalculate the r-value for `airbnb_density ~ rent_mean_both` for consistency.

In [ ]:
# create the log-transform from before
center_gdf["log_airbnb_density"] = np.log(center_gdf["airbnb_density"])

# naive correlation baseline (repetition)
naive_r, naive_p = stats.pearsonr(center_gdf["rent_mean_both"], center_gdf["log_airbnb_density"])
print(f"Naive correlation (Airbnb density ~ Rent): r = {naive_r:.3f}, p = {naive_p:.4f}\n")

If we square this resulting pearson r-value of 0.759 we get a $R^2$ of 0.576, which is almost identical to the value of 0.577 from before. With that out of the way, we can continue with calculating the residuals and then the pearson r for the correlation between these residuals:

In [ ]:
from src.comparative import get_part_corr
from src.comparative import plot_bar_r

y1_col = "rent_mean_both"
y2_col = "log_airbnb_density"

results = get_part_corr(center_gdf, y1_col, y2_col, naive_r, naive_p, n_boot = 10000)

In [ ]:
plot_bar_r(naive_r, ci_r2[0], ci_r2[1], results)

What the plot and the summary show us:  
- The r-value decreases for all tested city centers when removing their influence on both prediction and response variable.
- The largest decrease in r-value stems from using Grossmünster as the city center. This reduces the r-value by almost 50% from 0.759 to 0.420.
- Using the main station as the city center causes the smallest decrease.
- Yet, all p-values for the correlation between `log_airbnb_density ~ rent_mean_both` once the effect of distance to city center was removed are below 0.05, which means the correlation is significant.
- 
#### 5.4.4. Moran's I with confounding variable
We tried to **implement possible spatial structures** in our analysis by **including the distance to the city center** into our analysis, after finding **significant autocorrelation** in the data before correcting for distance to city center. Now that we have implemented this spatial variable of distance, we could hope for less spatial autocorrelation. However, there might still be remaining spatial structure in the residuals (distance to the city center captures only a first-order spatial trend (centrality effect), whereas spatial autocorrelation as measured by Moran’s I captures more general local spatial dependence. Therefore, even after controlling for distance, residual spatial structure may persist). To assess this, we compute Moran's I again but this time on the residuals of the regression between rental prices and log(Airbnb density) after we remove the parts of the variance explained by distance to city center..  
The null hypothesis of Moran’s I is that residuals are spatially randomly distributed. A significant result would indicate spatial autocorrelation, suggesting that the model does not fully capture spatial dependence in rental prices.

In [ ]:
from src.comparative import get_residuals
w = get_morans_weights(center_gdf)

y1_col = "rent_mean_both"
y2_col = "log_airbnb_density"
dist_variables = [
    "dist_hb",
    "dist_paradeplatz",
    "dist_bellevue",
    "dist_grossmunster"
]

results = []

for d in dist_variables:
    dist = center_gdf[d]

    # residualize each variable separately
    resid_rent = get_residuals(center_gdf[y1_col], dist)
    resid_airbnb = get_residuals(center_gdf[y2_col], dist)

    np.random.seed(42)

    # Moran's I for rent residuals
    moran_rent = Moran(resid_rent, w)
    # Moran's I for airbnb residuals
    moran_airbnb = Moran(resid_airbnb, w)

    results.append({
        "distance_center": d,
        "moran_rent_I": moran_rent.I,
        "moran_rent_p": moran_rent.p_sim,
        "moran_airbnb_I": moran_airbnb.I,
        "moran_airbnb_p": moran_airbnb.p_sim,
    })

results_df = pd.DataFrame(results)

results_df

The results suggest that rental prices and Airbnb density follow different spatial structures within Zurich. Rental prices are largely explained by a global center-periphery gradient, as spatial autocorrelation mostly disappears after controlling for distance to the city center. In contrast, Airbnb density retains significant local spatial clustering even after removing centrality effects, indicating that Airbnb listings are concentrated in specific neighborhoods beyond the general urban core. This difference in spatial structure helps explain why the correlation between Airbnb density and rents decreases substantially after controlling for distance, but does not disappear entirely. While part of the original association is attributable to shared centrality, the remaining correlation likely reflects localized neighborhood characteristics that are associated with both higher Airbnb activity and higher rental prices.
#### 5.4.3. Comparative analysis conclusion
Squaring the r-values yields the percentage of explained variance, which should be easier to interpret. The naive correlation has an $R^2$ of 0.579. This decreases steadily until the correlation correcting for distance to Grossmünster only has an $R^2$ value of 0.176. This means more than half the variation in rental prices was statistically associated with log(airbnb density) before correcting for distance to city center, and only 17.6% (which is still significant) of rental price variance is statistically associated with Airbnb density after correcting for distance to Grossmünster.

The naive regression suggests a moderate to strong positive association between Airbnb density and rental prices. This relationship is substantially reduced after controlling for distance to the city center, indicating that a large share of the observed correlation is driven by a shared spatial gradient. The largest decrease was observed with the city center defined as Grossmünster. However, a moderate and statistically significant association persists across all center definitions, suggesting that Airbnb density and rent are not fully explained by spatial proximity alone.

The Moran’s I analysis provides additional context for interpreting the observed relationship between Airbnb density and rental prices. Rental prices initially exhibit significant spatial autocorrelation, but this largely disappears after controlling for distance to the city center, suggesting that rent patterns are primarily driven by a global centrality gradient. In contrast, Airbnb density retains significant residual spatial autocorrelation even after controlling for distance, indicating localized neighborhood clustering beyond simple proximity to the center. Importantly, the correlation between Airbnb density and rental prices remains statistically significant after accounting for distance, suggesting that the observed association cannot be fully explained by shared spatial centrality alone.  

We find sufficient evidence to support H₁ — a significant positive correlation between `log(airbnb_density)` and `rent_mean_both` exists across all model specifications. However, a large share of this association is attributable to shared spatial centrality rather than a direct relationship between the two variables. The remaining partial correlation, while statistically significant, is moderate at best and should not be interpreted causally.  
The differing spatial structures revealed by Moran's I help explain why controlling for distance to city center substantially reduces the correlation: rental prices follow a global centrality gradient, whereas Airbnb density exhibits more localised neighbourhood clustering. This means the two variables are only partially driven by the same spatial process, which accounts for much of the drop in r² across center definitions.